# Spark-X2.6-4B-En4B-A1B (`v0.3`) — Formal All-in-One Full-Sequence Training, Stage-Merging `mHC` Looped Transformer, Self-Balancing `MoE-Engram` & Frontier Evaluation Suite

**End-to-End Reproducible Pipeline (`36 Physical Layers -> 72 Effective Layers`, `8.34B Total / 5.33B Active Parameters`)**

This formal all-in-one notebook covers the complete lifecycle of **`Spark-X2.6-4B-En4B-A1B` (`v0.3`)**:
1. **Pre-Training / Architectural Initialization & 5-Invariant Step-0 Verification**:
   - Wraps post-trained **`XHToken/Spark-X2.6-4B`** (`36L`, `4.23B` dense) with **24 `TwoStreamMHC` Doubly-Stochastic Sinkhorn Recurrence Controllers** (`Prelude [0..4) -> Pass 1 [4..28) -> Pass 2 [4..28) -> Pass 3 [12..24) -> Coda [28..36) = 72 Effective Layers`) and **8 `ZipfianTieredMoEEngram` Anchors** (`Layers 3, 7, 11, 15, 19, 23, 27, 31`; `4.11B` Total / `1.10B` Active).
   - Verifies exact Step-0 zero-gated identity (`max_abs_diff == 0.0` between `36L`, `60L`, and `72L`), causal n-gram invariance, and Birkhoff-Sinkhorn doubly-stochastic bounds ($\|M\|_2 \le 1$).
2. **Modifiable `MoE-Engram` & Auto-Detect / Self-Balancing Router**:
   - Fully customizable `MoE-Engram` hyperparameters (`ENG_EXPERTS`, `ENG_INITIAL_ACTIVE_EXPERTS`, `ENG_TOP_K`, `ENG_DIM`, `ENG_SHARED_RANK`, `ENG_EXPERT_RANK`, `ENG_HOT_BUCKETS`, `ENG_COLD_BUCKETS`, `ENG_PASS_GAIN_INIT`).
   - Hardware auto-detector (`auto_configure_vram_16k`) + **X-MoE Hypersphere Cosine Router** with **DeepSeek-V3 Auxiliary-Loss-Free Adaptive Load Balancing** (`expert_load_ema`, `expert_bias`, starved-centroid re-anchoring, and ST-MoE `z_loss`).
3. **6-Pillar Multi-Domain Dataset Curation & 16K Zero-Truncation Sequence Packing**:
   - Streams and filters 48+ verified shards across **STEM & PhD Science**, **Competition Math**, **SWE & Algorithmic Coding**, **Web Design & 3D WebGL**, **Claude Fable / Claude Opus 4.6–4.8 / GPT-6 Astra / GPT-5.4 Long-Run Reasoning CoT**, **Literacy & Multi-Turn Instruction Following**, and **Agentic Tool Calling**.
4. **Progressive 4-Phase Pre-Training to Post-Training Curriculum**:
   - **Stage 1 (Phase 1)**: `72L` `TwoStreamMHC` Looped Transformer adaptation (Backbone frozen, Engram off).
   - **Stage 2 (Phase 2A)**: `8-Expert Top-2` Coarse `MoE-Engram` + Self-Balancing Router formation at `8K`.
   - **Stage 3 (Phase 2B)**: Coarse-to-Fine Expert Upcycling (`8 -> 32` Experts, `Top-7`) at `16K` context.
   - **Stage 4 (Phase 3)**: Stage-merged Joint Full-Finetuning (`36L` Backbone at `0.25x` LR + `24x mHC` + `8x MoE-Engram-32`) at `16K`.
   - **Phase 4 (Post-Training Consolidation)**: Backbone-frozen Joint `mHC` + `MoE-Engram` finalization with **Per-Pass Telescoping Gates (`[P1: 0.35, P2: 0.15, P3: 0.05]`)** and a **Monotonic Held-Out Perplexity Gate**.
5. **Standalone Stage-Merged Checkpoint Export, Quantitative Ablation & 8-Benchmark Frontier Scorecard**:
   - Exports a self-contained Hugging Face repository (`TassP0lar/Spark-X2.6-4B-En4B-A1B`) and evaluates on **`SWE-bench Pro`**, **`SWE-bench Verified`**, **`GPQA Diamond`**, **`HLE`**, **`IFEval`**, **`IFBench`**, **`HumanEval`**, and **`MATH-500`**.


In [ ]:
# ==============================================================================
# Cell 1: Clone / Sync Repository & Install 16K Unsloth + Triton + Cut-CE + Liger Stack
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

pip_pkgs = [
    'torch>=2.5.0',
    'triton>=3.1.0',
    'transformers==4.57.1',
    'datasets>=3.2.0',
    'accelerate>=1.2.0',
    'safetensors>=0.4.5',
    'huggingface_hub>=0.27.0',
    'bitsandbytes>=0.45.0',
    'cut-cross-entropy',
    'liger-kernel',
    'unsloth',
    'openai>=1.58.0',
    'nltk',
    'langdetect',
    'psutil',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pip_pkgs], check=False)
print('Working directory synced:', Path.cwd())


In [ ]:
# ==============================================================================
# Cell 2: Modifiable MoE-Engram, mHC & 6-Pillar Curriculum Config + Hardware Auto-Detector
# ==============================================================================
import importlib, os
import sparkx_loop_v03_core
import sparkx_loop_v03_train_env
import sparkx_loop_v03_benchmarks
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
importlib.reload(sparkx_loop_v03_benchmarks)

from sparkx_loop_v03_core import (
    LoopV03Settings,
    SparkXLoopV03,
    count_en4b_a1b_params,
    patch_spark_for_unsloth_triton,
)
from sparkx_loop_v03_train_env import (
    DEFAULT_MAX_NEW_TOKENS,
    SparkXAgenticWorkspace,
    auto_configure_vram_16k,
    build_16k_stage_pools,
    build_default_cfg,
    curate_all_pillars_v03,
    generate_opencode_go_trajectories,
    run_step0_verification_suite,
    run_v03_four_stage_training,
    set_loglinear_kv_runtime,
)

def _get_secret(name: str):
    val = os.environ.get(name)
    if val:
        return val.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return None

HF_TOKEN = _get_secret('HF_TOKEN')
OPENCODE_API_KEY = _get_secret('OPENCODE_API_KEY')

# ------------------------------------------------------------------------------
# User-Modifiable Architecture, MoE-Engram, Self-Balancing Router & Stage Config
# ------------------------------------------------------------------------------
CFG = build_default_cfg()

# 1. 72-Layer Telescoping mHC Looped Transformer Geometry
CFG['LOOP_START'] = 4                                      # Prelude [0..4), Recurrent Core [4..28), Coda [28..36)
CFG['LOOP_END'] = 28
CFG['P3_START'] = 12                                       # Pass-3 Deep Deliberation Core [12..24) -> 72 Effective Layers
CFG['P3_END'] = 24
CFG['ENABLE_STAGE0_PROGRESSIVE_DEPTH_WARMUP'] = False      # Set True to prepend 48L -> 60L stage-merging warmup before 72L

# 2. Modifiable 8-Anchor 3-Tier Zipfian MoE-Engram & Self-Balancing Router Parameters
CFG['ENG_STRIDE'] = 4                                      # 8 Anchors at Layers 3, 7, 11, 15, 19, 23, 27, 31
CFG['ENG_HOT_BUCKETS'] = 16384                             # Tier-1A GPU HBM Hot N-gram Buckets (x3 scales)
CFG['ENG_COLD_BUCKETS'] = 49152                            # Tier-1B Host DRAM Cold N-gram Buckets (x3 scales)
CFG['ENG_DIM'] = 512                                       # Lexical & Expert Query Bottleneck Dimension
CFG['ENG_SHARED_RANK'] = 5888                              # Tier-2 100%-Active Shared SwiGLU Intermediate Rank
CFG['ENG_EXPERTS'] = 32                                    # Tier-3 Total Routed Experts per Anchor (256 across 8 anchors)
CFG['ENG_INITIAL_ACTIVE_EXPERTS'] = 8                      # Stage-2 Coarse Experts before 8 -> 32 Upcycling
CFG['ENG_TOP_K'] = 7                                       # Active Routed Experts per Token after Upcycling
CFG['ENG_INITIAL_TOP_K'] = 2                               # Active Routed Experts per Token in Stage 2
CFG['ENG_EXPERT_RANK'] = 3072                              # SwiGLU Intermediate Rank per Routed Expert
CFG['ENG_LOAD_EMA_DECAY'] = 0.05                           # Self-Balancing Router EMA Decay
CFG['ENG_BALANCE_STEP_SIZE'] = 0.0015                      # Adaptive Load-Balance Bias Step + Starved-Centroid Re-anchoring
CFG['ENG_USE_PASS_GAIN_SCHEDULE'] = True                   # Per-Pass Telescoping Gates [Pass1: 0.35, Pass2: 0.15, Pass3: 0.05]

# 3. 6-Pillar Multi-Domain Dataset Mixture Ratios
CFG['TARGET_TOTAL_SAMPLES'] = 2400
CFG['PILLAR_RATIOS'] = {
    'code_swe': 0.28,              # SWE Unified Diffs, Algorithmic Python/C++/JS, Magicoder, Opus 4.6-4.8 Code
    'math_stem': 0.24,             # PhD STEM (OpenScienceReasoning-2, MegaScience, Spark-234K, Kimi-PHD) & Competition Math
    'web_3d_ui': 0.16,             # Web Design, React/HTML5/CSS3 UI & 3D Three.js WebGL Synthesis
    'fable_opus_astra_cot': 0.14,  # Long-Run Reasoning CoT (GPT-6 Astra, GPT-5.4, Claude Fable 5.1, Opus 4.8, R1/OR1)
    'literacy_dialogue': 0.08,     # Literacy, Academic Prose, IFEval Constraint Following & Multi-Turn Dialogue
    'tool_agent': 0.10,            # Multi-Turn Agentic <tool_call> / <tool_response> Trajectories
}

# 4. Phase-4 Post-Training Consolidation Toggle
CFG['ENABLE_PHASE4_POST_TRAIN'] = True
CFG['PHASE4_STEPS'] = 80
CFG['PHASE4_TRAIN_MHC'] = True

# Auto-detect GPU VRAM & System RAM and configure optimal 16K execution flags
CFG = auto_configure_vram_16k(CFG)


In [ ]:
# ==============================================================================
# Cell 3: Instantiate Clean Post-Trained Spark-X2.6-4B + 72L mHC Loop + 8-Anchor MoE-Engram
#         & Run Step-0 5-Invariant Mathematical Verification Suite
# ==============================================================================
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

if 'tokenizer' not in globals() or tokenizer is None:
    print(f"Loading tokenizer from {CFG['BASE_MODEL_ID']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG['BASE_MODEL_ID'], trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

if 'model' in globals() and model is not None:
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if 'base_model' not in globals() or base_model is None or not getattr(base_model, '_sparkx_clean_posttrained', False):
    if 'base_model' in globals() and base_model is not None:
        del base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"Loading clean post-trained checkpoint from {CFG['BASE_MODEL_ID']}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG['BASE_MODEL_ID'],
        trust_remote_code=True,
        dtype=dtype,
    ).to(device)
    base_model._sparkx_clean_posttrained = True

patch_info = patch_spark_for_unsloth_triton(base_model)
print('Applied Unsloth/Triton backbone patches:', patch_info)

settings = LoopV03Settings(
    start=CFG['LOOP_START'],
    end=CFG['LOOP_END'],
    passes=2,
    p3_start=CFG['P3_START'],
    p3_end=CFG['P3_END'],
    enable_p3=True,
    use_mhc=True,
    use_engram=True,
    checkpoint_layers=True,
    offload_checkpoint_to_cpu=CFG['OFFLOAD_CHECKPOINT_CPU'],
    eng_hot_buckets=CFG['ENG_HOT_BUCKETS'],
    eng_cold_buckets=CFG['ENG_COLD_BUCKETS'],
    eng_dim=CFG['ENG_DIM'],
    eng_shared_rank=CFG['ENG_SHARED_RANK'],
    eng_experts=CFG['ENG_EXPERTS'],
    eng_initial_active_experts=CFG['ENG_INITIAL_ACTIVE_EXPERTS'],
    eng_top_k=CFG['ENG_TOP_K'],
    eng_initial_top_k=CFG['ENG_INITIAL_TOP_K'],
    eng_expert_rank=CFG['ENG_EXPERT_RANK'],
    eng_route_dim=CFG['ENG_ROUTE_DIM'],
    backbone_engram_stride=CFG['ENG_STRIDE'],
    eng_offload_tier1b_cpu=CFG['ENG_OFFLOAD_TIER1B_CPU'],
    eng_z_loss_coeff=CFG['ENG_Z_LOSS_COEFF'],
    ce_chunk_tokens=CFG['CE_CHUNK_TOKENS'],
)
model = SparkXLoopV03(base_model, settings=settings).to(device)
model.controllers.to(device=device)
model.engram.to(device=device)
model.backbone_engrams.to(device=device)

param_stats = count_en4b_a1b_params(model)
print('=' * 98)
print(
    f"[Parameter Breakdown] Total: {param_stats['model_total_params']/1e9:.2f}B | "
    f"Active/Token (Stage 2 E={model.engram.active_experts}, Top-{model.engram.active_top_k}): {param_stats['model_active_params']/1e9:.2f}B | "
    f"Backbone: {param_stats['backbone_total']/1e9:.2f}B | mHC Controllers: {param_stats['controllers_total']/1e6:.2f}M | "
    f"MoE-Engram Total: {param_stats['engram_total']/1e9:.2f}B (GPU: {param_stats['engram_gpu_total']/1e9:.2f}B, DRAM Tier-1B: {param_stats['engram_cpu_tier1b_total']/1e9:.2f}B)"
)
print('=' * 98)

# Verify all 5 Step-0 mathematical invariants (including exact 0.0 36L vs 60L vs 72L parity)
audit_metrics = run_step0_verification_suite(model, base_model, tokenizer)


In [ ]:
# ==============================================================================
# Cell 4: Stream 6-Pillar Multi-Domain Datasets (STEM, Math, SWE, Web/3D, Fable/Opus/Astra CoT,
#         Literacy & Tool Use), Run OpenCode Go Teacher & Pack 16K Zero-Truncation STAGE_POOLS
# ==============================================================================
import importlib, subprocess
import sparkx_loop_v03_train_env
subprocess.run(['git', 'pull', '--ff-only'], check=False)
importlib.reload(sparkx_loop_v03_train_env)
from sparkx_loop_v03_train_env import (
    build_16k_stage_pools,
    curate_all_pillars_v03,
    generate_opencode_go_trajectories,
)

curated_records = curate_all_pillars_v03(CFG)
teacher_records = generate_opencode_go_trajectories(CFG, api_key=OPENCODE_API_KEY)
all_records = curated_records + teacher_records

STAGE_POOLS = build_16k_stage_pools(all_records, tokenizer, CFG)


In [ ]:
# ==============================================================================
# Cell 5: Execute Full Progressive Curriculum (Pre-Training -> Stage-Merging mHC ->
#         MoE-Engram 8->32 Upcycling & Self-Balancing Routing -> Joint Full-FT -> Phase 4 Post-Train)
# ==============================================================================
import importlib, subprocess, sparkx_loop_v03_core, sparkx_loop_v03_train_env, sparkx_loop_v03_benchmarks
subprocess.run(['git', 'pull', '--ff-only'], check=False)
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
importlib.reload(sparkx_loop_v03_benchmarks)
from sparkx_loop_v03_train_env import run_v03_four_stage_training

train_summary = run_v03_four_stage_training(
    model=model,
    tokenizer=tokenizer,
    stage_pools=STAGE_POOLS,
    cfg=CFG,
    hf_token=HF_TOKEN,
)
print(f"Training & Phase-4 Consolidation complete at global_step={train_summary['global_step']} | Exported to: {train_summary['final_dir']}")


In [ ]:
# ==============================================================================
# Cell 6: Standalone Phase-4 Joint mHC + MoE-Engram Post-Training Consolidation
#         & Self-Balancing Router / 72L mHC Telemetry Audit
#   (Can also be run independently on any loaded checkpoint to refine mHC + MoE-Engram)
# ==============================================================================
import importlib
import torch
import sparkx_loop_v03_core
import sparkx_loop_v03_benchmarks as bench
importlib.reload(sparkx_loop_v03_core)
importlib.reload(bench)

# If Phase 4 was not already run inside Cell 5 (or if you want extra post-training consolidation steps):
EXTRA_PHASE4_STEPS = 0 if train_summary.get('phase4_report') is not None else int(CFG.get('PHASE4_STEPS', 80))
if EXTRA_PHASE4_STEPS > 0:
    phase4_report = bench.run_phase4_engram_consolidation(
        model=model,
        tokenizer=tokenizer,
        num_steps=EXTRA_PHASE4_STEPS,
        lr_engram=float(CFG.get('LR_ENGRAM', 1.5e-4)),
        lr_gates=5.0e-4,
        lr_mhc=3.0e-5,
        train_mhc=bool(CFG.get('PHASE4_TRAIN_MHC', True)),
        max_seq_len=768,
        eval_every=20,
        save_dir='/content/sparkx_v03_phase4_consolidated',
        push_to_hub=bool(CFG.get('PUSH_TO_HUB', False)),
        hub_repo_id=CFG['HF_REPO_ID'],
        token=HF_TOKEN,
    )

# Print 8-Anchor MoE-Engram Self-Balancing Router & Per-Pass Telescoping Gate Telemetry
rt = model.get_engram_routing_telemetry()
print('=' * 108)
print(
    f"[8-Anchor MoE-Engram Self-Balancing Router Summary] "
    f"Active Experts: {rt['active_experts']} (Top-K={rt['active_top_k']}) | "
    f"Mean Load CV: {rt['mean_load_cv']:.4f} | Min/Max Load Ratio: [{rt['min_load_ratio']:.2f}x, {rt['max_load_ratio']:.2f}x]"
)
print('-' * 108)
for anchor_key, info in rt['per_anchor'].items():
    print(
        f"  Anchor {anchor_key:3s} | E={info['active_experts']:2d}, Top-{info['active_top_k']} | "
        f"Load CV={info['load_cv']:.4f} [{info['min_load_ratio']:.2f}x..{info['max_load_ratio']:.2f}x] | "
        f"Temp={info['router_temp']:.4f} | Gate={info['gain_sigmoid']:.4f} | "
        f"PassScales={info['pass_scales']} | NGramWeights={info['ngram_weights']}"
    )
print('=' * 108)


In [ ]:
# ==============================================================================
# Cell 7: Quantitative 4-Way Perplexity/Accuracy Ablation + 3-Mode OOD Collapse Audit
#         + Unclamped 32K Agentic 3D WebGL Synthesis (`Neon Horizon 2099`)
# ==============================================================================
import importlib
import sparkx_loop_v03_core, sparkx_loop_v03_train_env, sparkx_loop_v03_benchmarks as bench
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
importlib.reload(bench)
from sparkx_loop_v03_train_env import DEFAULT_MAX_NEW_TOKENS, SparkXAgenticWorkspace, set_loglinear_kv_runtime

# 1. Quantitative Held-Out Perplexity & Next-Token Accuracy Delta (72L No-Engram vs 72L Full MoE-Engram)
ppl_delta = bench.evaluate_engram_perplexity_delta(model, tokenizer)
print('=' * 104)
print('[QUANTITATIVE HELD-OUT PERPLEXITY & ACCURACY AUDIT]')
print(
    f"  72L Loop + mHC Only (Engram OFF) : CE = {ppl_delta['no_engram_ce']:.5f} | PPL = {ppl_delta['no_engram_ppl']:.4f} | Top-1 Acc = {ppl_delta['no_engram_acc']:.2f}%\n"
    f"  72L Full + 32-Exp MoE-Engram ON  : CE = {ppl_delta['with_engram_ce']:.5f} | PPL = {ppl_delta['with_engram_ppl']:.4f} | Top-1 Acc = {ppl_delta['with_engram_acc']:.2f}% "
    f"(Delta_PPL = {ppl_delta['delta_ppl']:+.4f})"
)
print('=' * 104)

# 2. 3-Mode OOD Collapse Audit (`dijkstra_grid`) + Unclamped 32K 3D WebGL Synthesis
set_loglinear_kv_runtime(model, enabled=True, exact_threshold=32768)
workspace = SparkXAgenticWorkspace('/content/sakura_agent_workspace_v03', model=model, tokenizer=tokenizer)
mission_result = workspace.run_agentic_mission(
    'Build Neon Horizon 2099 3D Cyberpunk Hovercar Flight Simulator',
    target_filename='index.html',
    max_turns=3,
    max_new_tokens_per_turn=DEFAULT_MAX_NEW_TOKENS,
    run_ablation_first=True,
    use_ood_mission=True,
)


In [ ]:
# ==============================================================================
# Cell 8: Frontier Benchmark Scorecard (`SWE-bench Pro`, `SWE-bench Verified`,
#         `GPQA Diamond`, `HLE`, `IFEval`, `IFBench`, `HumanEval`, `MATH-500`)
# ==============================================================================
import importlib
import sparkx_loop_v03_benchmarks as bench
importlib.reload(bench)

frontier_report = bench.run_full_frontier_benchmark_suite(
    model=model,
    tokenizer=tokenizer,
    samples_ifeval=25,
    samples_ifbench=25,
    samples_gpqa=25,
    samples_hle=15,
    samples_swe_pro=10,
    samples_swe_verified=10,
    samples_humaneval=25,
    samples_math500=25,
    output_dir='/content/sparkx_v03_benchmark_results',
    token=HF_TOKEN,
)
